# PsychQwen3 — SFT Qwen3-8B на Psychiatry ReAct датасет

Датасет: `react_sft_10k.jsonl` (10 000 ReAct-трейсов, психиатрия, русский).

Ноутбук:
1. **Cell 1** — загрузка train/val, модели Qwen3-8B, выключение thinking, форматирование в SFT-текст.
2. **Cell 2** — LoRA SFT на train с eval на val; финальный адаптер сохраняется на Google Drive в `/MyDrive/psychqwen3/` (чекпоинты внутри обучения НЕ сохраняются).
3. **Cell 3** — инференс и тест (sanity-check + опциональный MCQ accuracy на val).

In [ ]:
# Cell 1: загрузка датасетов, модели, выключение thinking, форматирование jsonl → SFT-формат

import json, os
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ================== CONFIG ==================
DRIVE_DIR   = "/content/drive/MyDrive/psychqwen3"      # папка на Google Drive для адаптеров
TRAIN_PATH  = "/content/train.jsonl"
VAL_PATH    = "/content/val.jsonl"
MODEL_NAME  = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LEN = 4096     # p90 примера ~1.3k токенов, но react_loop бывают длинными

# ================== 1. Mount Drive ==================
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive mounted, output dir:", DRIVE_DIR)

# ================== 2. Загрузка датасетов (train/val уже разбиты 95/5) ==================
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw_train = load_jsonl(TRAIN_PATH)
raw_val   = load_jsonl(VAL_PATH)
print(f"train: {len(raw_train)} | val: {len(raw_val)}")

# метаданные датасета (kind/difficulty/noisy) нам для SFT не нужны — только messages
from collections import Counter
kinds = Counter(ex["kind"] for ex in raw_train)
print("kinds:", dict(kinds))

# ================== 3. Модель ==================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    # token="hf_...",  # раскомментируй, если нужно
)

# ================== 4. Выключение thinking у Qwen3 ==================
# Qwen3 — гибридная модель. В режиме "thinking" перед ответом она пишет
# <think>...</think>. Наш датасет — свой формат рассуждений
# (Thought:/Action:/Observation:/Final Answer:) — <think> будет с ним конфликтовать.
# Решение: обучаем в no-think режиме. В тренировочном тексте к каждому assistant-ответу
# добавляем пустой <think>\n\n</think> — так шаблон понимает, что thinking выключен,
# и при инференсе модель не будет его генерировать (при enable_thinking=False).
tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

def add_nothink(example):
    msgs = []
    for m in example["messages"]:
        if m["role"] == "assistant":
            msgs.append({"role": "assistant", "content": "<think>\n\n</think>\n\n" + m["content"]})
        else:
            msgs.append({"role": m["role"], "content": m["content"]})
    return {"messages": msgs}

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )}

train_ds = Dataset.from_list(raw_train).map(add_nothink).map(to_text, remove_columns=[c for c in raw_train[0].keys() if c != "messages"])
val_ds   = Dataset.from_list(raw_val).map(add_nothink).map(to_text, remove_columns=[c for c in raw_val[0].keys() if c != "messages"])

# ================== 5. Проверка формата ==================
print("="*60)
print("TRAIN[0] text (первые 1200 символов):")
print("="*60)
print(train_ds[0]["text"][:1200])
print()
# sanity-check: считаем, сколько пустых <think>-блоков (должно быть = числу assistant-реплик)
n_think = sum(ex["text"].count("<think>\n\n</think>") for ex in train_ds)
n_asst  = sum(len([m for m in ex["messages"] if m["role"]=="assistant"]) for ex in train_ds)
print(f"empty think-blocks: {n_think} / assistant messages: {n_asst}  (должно совпадать)")

# токен-статистика (для выбора MAX_SEQ_LEN)
import numpy as np
lens = [len(tokenizer(ex["text"])["input_ids"]) for ex in train_ds.select(range(500))]
print(f"token len (500 примеров): p50={int(np.percentile(lens,50))}, p90={int(np.percentile(lens,90))}, p99={int(np.percentile(lens,99))}, max={max(lens)}")


In [ ]:
# Cell 2: LoRA (unsloth) + SFT на train/eval val + сохранение адаптера на Drive

from unsloth.chat_templates import SFTTrainer, train_on_responses_only
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch

# ================== CONFIG (экспериментируй) ==================
LEARNING_RATE = 5e-5     # история: 1e-4 училась, но была осторожность; 2e-6..6e-6 - адаптер не обучился (тесты на чистой базе). 5e-5 - рабочая точка.
NUM_EPOCHS    = 2        # следи за val_loss и trap-тестом из Cell 3
LORA_R        = 16
LORA_ALPHA    = 32       # обычно = 2*r
LORA_DROPOUT  = 0.0

# Чекпоинты пишем ЛОКАЛЬНО в /content (не на Drive — там медленный I/O),
# save_total_limit=1 оставляет только лучший — диск не распухает.
LOCAL_CKPT_DIR = "/content/checkpoints"

# ================== LoRA через unsloth ==================
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",   # экономит VRAM, позволяет длинный контекст
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

# ================== SFTTrainer (unsloth) ==================
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=4,
    packing=False,               # честный val_loss без паковки
    args=TrainingArguments(
        output_dir=LOCAL_CKPT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,               # эффективный batch = 16
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=250,          # eval на T4 медленный; можно увеличить интервал
        save_strategy="steps",
        save_steps=250,          # ДОЛЖНО совпадать с eval_steps (load_best требует)
        save_total_limit=1,      # на диске остаётся только ЛУЧШИЙ чекпоинт, не все
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=1.0,
        seed=3407,
        report_to="none",
        dataloader_num_workers=2,
    ),
)

# ================== Маска лосса: учим ТОЛЬКО ответы ассистента ==================
# Системный промпт и вопрос врача не попадают в лосс — модель не учится
# воспроизводить системные промпты, только генерить ответы.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# проверка маски на одном примере (decode: instruction_token_ids должны быть False)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["input_ids"]]).replace(tokenizer.pad_token, " ")

# ================== Train ==================
trainer_stats = trainer.train()

# ================== Сохранение ЛУЧШЕГО адаптера на Drive ==================
# load_best_model_at_end=True: после .train() в trainer.model уже загружен
# чекпоинт с минимальным eval_loss — сохраняем именно его.
import re as _re
best_ckpt = trainer.state.best_model_checkpoint   # путь к лучшему чекпоинту
best_loss = trainer.state.best_metric             # eval_loss лучшего
best_step = int(_re.search(r"checkpoint-(\d+)", best_ckpt).group(1)) if best_ckpt else trainer.state.global_step
print(f"best eval_loss={best_loss:.4f} на шаге {best_step} ({best_ckpt})")

SAVE_DIR = os.path.join(
    DRIVE_DIR,
    f"lora_sft_lr{LEARNING_RATE:.0e}_ep{NUM_EPOCHS}_r{LORA_R}_best{best_loss:.3f}",
)
os.makedirs(SAVE_DIR, exist_ok=True)
trainer.model.save_pretrained(SAVE_DIR)   # unsloth: только LoRA-адаптер
tokenizer.save_pretrained(SAVE_DIR)
print("Лучший адаптер сохранён в:", SAVE_DIR)

# подчистить локальные чекпоинты — на Drive они не нужны
import shutil
shutil.rmtree(LOCAL_CKPT_DIR, ignore_errors=True)
print("локальные чекпоинты удалены")

# ================== Отчёт ==================
m = trainer_stats.metrics
print(f"train_runtime: {m['train_runtime']:.1f}s | train_loss: {m['train_loss']:.4f}")
if torch.cuda.is_available():
    print(f"peak VRAM: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")



In [ ]:
# Cell 3: инференс и тест на val (no-think режим)

from unsloth import FastLanguageModel

# ================== Загружаем сохранённый адаптер ==================
# ВАЖНО: базовая модель должна быть та же, что при обучении
LOAD_DIR = SAVE_DIR  # или пропиши вручную, напр. "/content/drive/MyDrive/psychqwen3/lora_sft_lr1e-04_ep3_r16"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LOAD_DIR,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)  # 2x быстрее инференс

# ================== Промпт-хелпер ==================
def chat(user_msg, system_prompt=None, max_new_tokens=700, temperature=0.3):
    msgs = []
    if system_prompt:
        msgs.append({"role":"system","content":system_prompt})
    msgs.append({"role":"user","content":user_msg})
    # enable_thinking=False — форматированный ответ без <think>-блока
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return reply

# ================== Быстрый sanity-check ==================
SYS = "Ты — клинический ассистент психиатра. Отвечай структурированно, с указанием препаратов, дозировок и мониторинга. Аудитория — врачи. Если данных недостаточно — прямо скажи об этом."

print("="*60); print("ТЕСТ 1: старт терапии"); print("="*60)
print(chat("Пациент: мужчина 45 лет, депрессивный эпизод (F32.1), БАР в анамнезе (маниакальный эпизод 3 года назад). Чем лечить?", system_prompt=SYS))

print("="*60); print("ТЕСТ 2: trap-вопрос (ложная предпосылка)"); print("="*60)
print(chat("Пациент на литии 1200 мг начал пить ибупрофен от болей в спине. Через неделю тремор и жажда. Это не связано с литием, верно?", system_prompt=SYS))

print("="*60); print("ТЕСТ 3: abstention (недостаточно данных)"); print("="*60)
print(chat("Пациенту 40 лет с депрессией. Что назначить?", system_prompt=SYS))

# ================== Систематический прогон по val ==================
# MCQ accuracy: для kind == "mcq" парсим ответ "Правильный ответ: X" и сверяем с correct_idx
import re

def eval_mcq(n=100, temperature=0.2):
    mcq_items = [ex for ex in raw_val if ex["kind"]=="mcq"][:n]
    correct = 0
    results = []
    for ex in mcq_items:
        opts = "\n".join(f"{chr(65+i)}) {o[:200]}" for i, o in enumerate(ex["options"]))
        user = ex["messages"][1]["content"] + "\n\nВарианты ответа:\n" + opts + "\n\nОтветь только буквой правильного варианта."
        reply = chat(user, system_prompt=ex["messages"][0]["content"], max_new_tokens=300, temperature=temperature)
        m = re.search(r"([A-D])\b", reply.strip()[:50])
        pred = {"A":0,"B":1,"C":2,"D":3}.get(m.group(1), -1) if m else -1
        is_ok = pred == ex["correct_idx"]
        correct += is_ok
        results.append({"id":ex["id"],"pred":pred,"gold":ex["correct_idx"],"ok":is_ok})
    acc = correct/len(mcq_items)
    print(f"MCQ accuracy (n={len(mcq_items)}): {acc:.3f}")
    return results

# Запусти когда захочешь замерить (занимает ~2-3 мин на 20 примеров на T4)
# mcq_results = eval_mcq(n=50)


## Cell 4 — GGUF + Hugging Face

Слияние LoRA → квантование в GGUF (q4_k_m) → no-think chat template → upload на HF. Перед запуском: замени `HF_REPO` на свой, при необходимости задай `HF_TOKEN`.

In [ ]:
# Cell 4: конвертация в GGUF и загрузка на Hugging Face (thinking выключен в шаблоне)

# ================== CONFIG ==================
SAVE_DIR  = SAVE_DIR          # путь к LoRA-адаптеру с Drive (из Cell 2), напр.:
# SAVE_DIR = "/content/drive/MyDrive/psychqwen3/lora_sft_lr5e-06_ep1_r16_best0.350"
HF_REPO   = "voperl/PsychQwen3-8B-GGUF"   # <- замени на свой repo
HF_TOKEN  = None               # или вставь токен строкой (лучше через huggingface-cli login)
GGUF_QUANT = "q4_k_m"          # q4_k_m - стандарт для 8B на consumer железе
                          # варианты: q5_k_m (чуть лучше, крупнее), q6_k, q8_0 (почти fp16)
MAX_QLEN  = 4096               # длина контекста в GGUF-метаданных



# ================== 1. Сливаем LoRA обратно в базу (float16) ==================
# unsloth умеет сохранять merged GGUF напрямую из LoRA
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SAVE_DIR,
    max_seq_length=MAX_QLEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# ВАЖНО (thinking off в GGUF): у GGUF-модели нет параметра enable_thinking при инференсе.
# Этим управляет **шаблон чата**, зашитый в tokenizer_config.json.
# Мы ниже правим шаблон: принудительно вставляем пустой <think></think> в
# generation-prompt, чтобы модель сразу отвечала без рассуждений.
# (Так делают все «no-think» GGUF экспорты Qwen3.)

# ================== 2. Правим chat template: no-think по умолчанию ==================
TEMPLATE = """{%- if tools %}
{{- '<|im_start|>system\\n' }}
{%- if messages[0].role == 'system' %}
{{- messages[0].content + '\\n\\n' }}
{%- endif %}
{{- "# Tools\\n\\nYou may call one or more functions..." }}
{%- else %}
{%- if messages[0].role == 'system' %}
{{- '<|im_start|>system\\n' + messages[0].content + '<|im_end|>\\n' }}
{%- endif %}
{%- endif %}
{%- for message in messages %}
{%- if message.role == 'user' %}
{{- '<|im_start|>user\\n' + message.content + '<|im_end|>\\n' }}
{%- elif message.role == 'assistant' %}
{{- '<|im_start|>assistant\\n' }}
{%- set content = message.content %}
{%- set reasoning_content = '' %}
{%- if '</think>' in content %}
{%- set content = content.split('</think>')[-1].lstrip('\\n') %}
{%- endif %}
{{- '<think>\\n\\n</think>\\n\\n' + content + '<|im_end|>\\n' }}
{%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
{{- '<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n' }}
{%- endif %}"""

tokenizer.chat_template = TEMPLATE

# Проверка: как теперь выглядит промпт на генерацию
msgs = [{"role": "user", "content": "тест"}]
rendered = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
assert "<think>\n\n</think>" in rendered, "no-think префикс не вставился!"
print("Промпт с no-think (первые 300 симв.):")
print(rendered[:300])

# ================== 3. Сохраняем GGUF (unsloth: merge LoRA + квантование в один шаг) ==================
GGUF_PATH = SAVE_DIR.rstrip("/") + f"-{GGUF_QUANT}.gguf"
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method=GGUF_QUANT,
    maximum_lora_rank=None,
)
print("GGUF сохранён локально:", GGUF_PATH)

# ================== 4. Загрузка на HF ==================
from huggingface_hub import HfApi, login
if HF_TOKEN:
    login(HF_TOKEN)
else:
    login()   # спросит токен интерактивно (colab: popup/ввод в ячейку)

api = HfApi()
api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)

api.upload_file(
    path_or_fileobj=GGUF_PATH,
    path_in_repo=GGUF_PATH.split("/")[-1],
    repo_id=HF_REPO,
    repo_type="model",
    commit_message=f"PsychQwen3-8B SFT {GGUF_QUANT}, no-think chat template",
)
print(f"Загружено: https://huggingface.co/{HF_REPO}")

# ================== 5. (опц.) Модель-карточка ==================
card = f"""---
license: apache-2.0
base_model: unsloth/Qwen3-8B
tags:
- psychiatry
- medical
- russian
- qwen3
- lora
- sft
---

# PsychQwen3-8B ({GGUF_QUANT}, no-think)

Клинический ассистент психиатра (русскоязычный), Qwen3-8B + LoRA SFT
на 10k ReAct-трейсов психиатрии (react_sft_10k).

## Chat template
**Thinking выключен**: в шаблоне при генерации сразу подставляется пустой
`<think>\\n\\n</think>`. Модель отвечает сразу, без рассуждений.

## Использование (LM Studio / llama.cpp)
Просто загрузите GGUF — шаблон подхватится из `tokenizer_config.json`.
System prompt рекомендуется тот же, что в обучении:
"Ты — клинический ассистент психиатра. Отвечай структурированно..."
"""
with open(GGUF_PATH.rsplit("/", 1)[0] + "/README.md", "w", encoding="utf-8") as f:
    f.write(card)
api.upload_file(
    path_or_fileobj=GGUF_PATH.rsplit("/", 1)[0] + "/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO,
    repo_type="model",
    commit_message="add model card",
)
print("README загружен")
